# 🎯 Social Media Sentiment Analysis — Complete Pipeline

> **CUD Machine Learning Final Project** | End-to-end multi-model sentiment analysis

This notebook covers all 6 phases:
1. **Data Exploration** — Load, visualize, and understand the dataset
2. **Feature Engineering** — Text cleaning, TF-IDF, Word2Vec, tokenization
3. **Classical ML Models** — VADER, Logistic Regression, Random Forest, SVM
4. **Deep Learning (BiLSTM)** — Bidirectional LSTM with Attention mechanism
5. **Model Comparison** — Comprehensive benchmarking & visualization
6. **Live Inference** — Demo inference with all models

---
**Dataset:** `tweet_eval/sentiment` (HuggingFace)  
**Models:** VADER · Logistic Regression · Random Forest · SVM · BiLSTM+Attention  
**Classes:** Negative (0) · Neutral (1) · Positive (2)

## 📦 Install Dependencies

In [ ]:
# Run once to install all required packages
!pip install -q datasets transformers pandas numpy scikit-learn matplotlib seaborn \
    wordcloud vaderSentiment gensim tensorflow keras joblib plotly squarify

## ⚙️ Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import json
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from pathlib import Path

# Sklearn
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize

# VADER
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Keras / TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Plot style
plt.style.use('dark_background')
COLORS = {'positive': '#10b981', 'neutral': '#6b7280', 'negative': '#ef4444'}
MODEL_COLORS = ['#ef4444', '#3b82f6', '#f97316', '#8b5cf6', '#10b981']
sns.set_palette([COLORS['negative'], COLORS['neutral'], COLORS['positive']])

LABEL_MAP   = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL_NAMES = ['negative', 'neutral', 'positive']

print(f'TensorFlow: {tf.__version__}')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print('Environment ready ✓')

# Plotly for interactive visualisations
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import squarify

PLOTLY_TEMPLATE = 'plotly_dark'
PLOTLY_COLORS   = {'positive': '#10b981', 'neutral': '#6b7280', 'negative': '#ef4444'}


---
## 📊 Phase 1 — Data Exploration

Load the `tweet_eval/sentiment` dataset from HuggingFace and perform comprehensive exploratory data analysis.

In [ ]:
from datasets import load_dataset

print('Loading tweet_eval/sentiment from HuggingFace...')
try:
    raw = load_dataset('tweet_eval', 'sentiment')
    print('Dataset loaded from HuggingFace ✓')
except Exception as e:
    print(f'HuggingFace failed ({e}), using cardiffnlp/tweet_sentiment_multilingual as fallback...')
    raw = load_dataset('cardiffnlp/tweet_sentiment_multilingual', 'english')

# Convert to DataFrames
dfs = {}
for split in raw.keys():
    d = raw[split]
    df_s = pd.DataFrame({'text': d['text'], 'label': d['label']})
    df_s['label_name'] = df_s['label'].map(LABEL_MAP)
    df_s['split'] = split
    dfs[split] = df_s

df_all = pd.concat(dfs.values(), ignore_index=True)
df_train = dfs.get('train', df_all[df_all['split']=='train'])
df_test  = dfs.get('test',  df_all[df_all['split']=='test'])
df_val   = dfs.get('validation', df_all[df_all['split']=='validation'])

print(f'\nTotal samples  : {len(df_all):,}')
print(f'Train          : {len(df_train):,}')
print(f'Validation     : {len(df_val):,}')
print(f'Test           : {len(df_test):,}')
print(f'Columns        : {list(df_all.columns)}')
df_all.head()

In [ ]:
# ── Class distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Class Distribution Analysis', fontsize=16, fontweight='bold', color='white')

# 1. Bar chart — all splits
split_labels = pd.crosstab(df_all['split'], df_all['label_name'])[LABEL_NAMES]
split_labels.plot(kind='bar', ax=axes[0], color=[COLORS[l] for l in LABEL_NAMES],
                  edgecolor='none', width=0.7)
axes[0].set_title('Count by Split', color='white')
axes[0].set_xlabel('Split'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Sentiment', facecolor='#1f2937', labelcolor='white')

# 2. Pie chart — training set
counts = df_train['label_name'].value_counts().reindex(LABEL_NAMES).fillna(0)
axes[1].pie(counts, labels=LABEL_NAMES, autopct='%1.1f%%',
            colors=[COLORS[l] for l in LABEL_NAMES],
            startangle=90, textprops={'color':'white'})
axes[1].set_title('Training Set Distribution', color='white')

# 3. Count bars — overall
overall = df_all['label_name'].value_counts().reindex(LABEL_NAMES).fillna(0)
bars = axes[2].bar(LABEL_NAMES, overall, color=[COLORS[l] for l in LABEL_NAMES],
                   edgecolor='none', width=0.5)
for bar, val in zip(bars, overall):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{int(val):,}', ha='center', va='bottom', color='white', fontsize=10)
axes[2].set_title('Overall Class Counts', color='white')
axes[2].set_ylabel('Count')

for ax in axes:
    ax.set_facecolor('#111827')
fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

print('\nClass distribution (training):')
print(df_train['label_name'].value_counts().to_string())

In [ ]:
# ── Text length analysis ─────────────────────────────────────────────────────
df_all['char_len'] = df_all['text'].str.len()
df_all['word_count'] = df_all['text'].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Text Length Analysis', fontsize=16, fontweight='bold', color='white')

# Character length histogram
for label in LABEL_NAMES:
    data = df_all[df_all['label_name']==label]['char_len']
    axes[0].hist(data, bins=50, alpha=0.6, label=label, color=COLORS[label])
axes[0].set_title('Character Length Distribution', color='white')
axes[0].set_xlabel('Characters'); axes[0].set_ylabel('Frequency')
axes[0].legend(facecolor='#1f2937', labelcolor='white')

# Word count boxplot
data_by_class = [df_all[df_all['label_name']==l]['word_count'].values for l in LABEL_NAMES]
bp = axes[1].boxplot(data_by_class, labels=LABEL_NAMES, patch_artist=True,
                     medianprops={'color':'white', 'linewidth':2})
for patch, label in zip(bp['boxes'], LABEL_NAMES):
    patch.set_facecolor(COLORS[label])
    patch.set_alpha(0.7)
axes[1].set_title('Word Count by Class (Boxplot)', color='white')
axes[1].set_ylabel('Word Count')

# Summary stats table
axes[2].axis('off')
stats = df_all.groupby('label_name')[['char_len','word_count']].describe().round(1)
summary = []
for label in LABEL_NAMES:
    row = stats.loc[label]
    summary.append([label, int(row['char_len']['count']), f"{row['char_len']['mean']:.0f}",
                    f"{row['word_count']['mean']:.1f}", f"{row['char_len']['max']:.0f}"])
table = axes[2].table(
    cellText=summary,
    colLabels=['Class', 'Count', 'Avg Chars', 'Avg Words', 'Max Chars'],
    cellLoc='center', loc='center'
)
table.auto_set_font_size(False); table.set_fontsize(11)
table.scale(1, 2)
for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor('#374151')
    cell.set_facecolor('#1f2937' if r > 0 else '#374151')
    cell.set_text_props(color='white')
axes[2].set_title('Length Statistics per Class', color='white')

for ax in axes[:2]: ax.set_facecolor('#111827')
fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

In [ ]:
# ── Word clouds per class ────────────────────────────────────────────────────
try:
    from wordcloud import WordCloud
    has_wordcloud = True
except ImportError:
    has_wordcloud = False
    print('wordcloud not installed, skipping.')

if has_wordcloud:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('Word Clouds per Sentiment Class', fontsize=16, fontweight='bold', color='white')

    BG_COLORS = {'negative': '#1a0000', 'neutral': '#111827', 'positive': '#001a0a'}
    WC_COLORS = {'negative': 'Reds', 'neutral': 'Greys', 'positive': 'Greens'}

    for ax, label in zip(axes, LABEL_NAMES):
        text_corpus = ' '.join(df_train[df_train['label_name']==label]['text'].astype(str))
        wc = WordCloud(
            width=600, height=400, background_color=BG_COLORS[label],
            colormap=WC_COLORS[label], max_words=100,
            stopwords={'the','a','an','and','or','but','in','on','at','to','for',
                       'of','is','are','was','were','be','this','that','with'},
            random_state=SEED
        ).generate(text_corpus)
        ax.imshow(wc, interpolation='bilinear')
        ax.set_title(f'{label.capitalize()}', color=COLORS[label], fontsize=14, fontweight='bold')
        ax.axis('off')

    fig.patch.set_facecolor('#0f172a')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Top 20 words per class ───────────────────────────────────────────────────
STOPWORDS = {'the','a','an','and','or','but','in','on','at','to','for','of','is',
             'are','was','were','be','this','that','with','i','it','my','me','you',
             'we','they','have','has','had','not','do','did','he','she','so','if',
             'can','will','just','get','got','all','no','what','your','from','by'}

def top_words(texts, n=20):
    words = []
    for t in texts:
        words.extend([w.lower() for w in re.findall(r'[a-zA-Z]+', str(t))
                      if w.lower() not in STOPWORDS and len(w) > 2])
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Top 20 Words per Sentiment Class', fontsize=16, fontweight='bold', color='white')

for ax, label in zip(axes, LABEL_NAMES):
    top = top_words(df_train[df_train['label_name']==label]['text'])
    words, counts = zip(*top)
    bars = ax.barh(list(reversed(words)), list(reversed(counts)),
                   color=COLORS[label], alpha=0.8, edgecolor='none')
    ax.set_title(f'{label.capitalize()}', color=COLORS[label], fontsize=13)
    ax.set_xlabel('Frequency')
    ax.set_facecolor('#111827')

fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

In [ ]:
# ── Data quality report ──────────────────────────────────────────────────────
print('=== DATA QUALITY REPORT ===')
print(f"Total rows          : {len(df_all):,}")
print(f"Missing texts       : {df_all['text'].isna().sum()}")
print(f"Empty texts         : {(df_all['text'].str.strip() == '').sum()}")
print(f"Duplicate texts     : {df_all['text'].duplicated().sum():,}")
print(f"Avg text length     : {df_all['char_len'].mean():.1f} chars")
print(f"Median text length  : {df_all['char_len'].median():.0f} chars")
print(f"Max text length     : {df_all['char_len'].max()} chars")
print(f"Min text length     : {df_all['char_len'].min()} chars")
print(f"\nClass balance (train):")
vc = df_train['label_name'].value_counts()
for label, count in vc.items():
    pct = count / len(df_train) * 100
    print(f"  {label:10s}: {count:6,}  ({pct:.1f}%)")

### 🔤 Bigram Analysis & Word-Frequency Treemaps
Top bigrams per class visualised as Plotly treemaps — shows phrase-level patterns.

In [ ]:
# ── Bigram frequency per class → Plotly treemap ─────────────────────────
from sklearn.feature_extraction.text import CountVectorizer

STOP = {'the','a','an','and','or','but','in','on','at','to','for','of','is','are',
        'was','were','be','this','that','with','i','it','my','me','you','we','they',
        'have','has','had','not','do','did','he','she','so','if','can','will','just',
        'get','got','all','no','what','your','from','by','url','user'}

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=['😞 Negative', '😐 Neutral', '😊 Positive'],
                    specs=[[{'type': 'treemap'}]*3])

for col_idx, label in enumerate(LABEL_NAMES, 1):
    corpus = df_train[df_train['label_name']==label]['clean_text'].dropna().tolist()
    cv = CountVectorizer(ngram_range=(2,2), max_features=30, stop_words=list(STOP))
    try:
        cv.fit(corpus)
        counts  = cv.transform(corpus).sum(axis=0).A1
        phrases = cv.get_feature_names_out()
        top_idx = counts.argsort()[-20:][::-1]
        labels_list = [phrases[i] for i in top_idx]
        values_list = [int(counts[i]) for i in top_idx]
        color_hex = {'positive':'#10b981','neutral':'#6b7280','negative':'#ef4444'}[label]
        fig.add_trace(
            go.Treemap(labels=labels_list, parents=['']*len(labels_list),
                       values=values_list,
                       marker=dict(colorscale=[[0,'#111827'],[1,color_hex]],
                                   colors=values_list, showscale=False),
                       textfont=dict(size=13),
                       name=label),
            row=1, col=col_idx
        )
    except Exception as e:
        print(f'{label}: {e}')

fig.update_layout(template=PLOTLY_TEMPLATE, height=480,
                  title_text='Top 20 Bigrams per Sentiment Class (Treemap)',
                  title_font_size=16)
fig.show()


In [ ]:
# ── Tweet structural fingerprint per class ───────────────────────────────
import re

def fingerprint(df):
    return pd.Series({
        'Has URL (%)':     df['text'].str.contains(r'https?://', regex=True).mean() * 100,
        'Has Mention (%)': df['text'].str.contains(r'@\w+', regex=True).mean() * 100,
        'Has Hashtag (%)': df['text'].str.contains(r'#\w+', regex=True).mean() * 100,
        'Has Emoji (%)':   df['text'].str.contains(
            '[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF]',
            regex=True).mean() * 100,
        'Avg Words':       df['word_count'].mean(),
        'Avg Chars':       df['char_len'].mean(),
        'Exclamation (%)': df['text'].str.contains('!').mean() * 100,
        'Question (%)':    df['text'].str.contains('\\?').mean() * 100,
        'ALL CAPS words':  df['text'].apply(
            lambda t: sum(1 for w in str(t).split() if w.isupper() and len(w)>1)).mean(),
    })

fp = pd.DataFrame({l: fingerprint(df_train[df_train['label_name']==l]) for l in LABEL_NAMES})

# Radar / parallel-coords view
fig = go.Figure()
cat_cols = ['Has URL (%)','Has Mention (%)','Has Hashtag (%)','Has Emoji (%)',
            'Exclamation (%)','Question (%)','ALL CAPS words']
color_map = {'negative':'#ef4444','neutral':'#6b7280','positive':'#10b981'}

for label in LABEL_NAMES:
    vals = fp.loc[cat_cols, label].tolist()
    vals_closed = vals + [vals[0]]
    theta_closed = cat_cols + [cat_cols[0]]
    fig.add_trace(go.Scatterpolar(
        r=vals_closed, theta=theta_closed, fill='toself',
        name=label.capitalize(), line_color=color_map[label], opacity=0.7
    ))

fig.update_layout(template=PLOTLY_TEMPLATE, height=500,
                  polar=dict(radialaxis=dict(visible=True)),
                  title='Tweet Structural Fingerprint by Sentiment Class (Radar)')
fig.show()

# Bar chart — numeric stats
fig2 = px.bar(fp.loc[['Avg Words','Avg Chars','ALL CAPS words']].T.reset_index(),
              x='label_name', y=['Avg Words','Avg Chars','ALL CAPS words'],
              barmode='group', template=PLOTLY_TEMPLATE,
              color_discrete_sequence=['#6366f1','#3b82f6','#f97316'],
              title='Avg Words / Chars / ALL-CAPS words per Class')
fig2.update_xaxes(title=''); fig2.update_yaxes(title='Mean count')
fig2.show()

print(fp.round(2).to_string())


---
## ⚙️ Phase 2 — Feature Engineering

Clean text, build TF-IDF features, and tokenize for deep learning.

In [ ]:
# ── Text cleaning ────────────────────────────────────────────────────────────
URL_RE      = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE  = re.compile(r'@\w+')
HASHTAG_RE  = re.compile(r'#(\w+)')
EMOJI_RE    = re.compile('[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF'
                         '\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF'
                         '\U00002702-\U000027B0\U000024C2-\U0001F251]+', flags=re.UNICODE)
REPEAT_RE   = re.compile(r'(.)\1{2,}')
SPACE_RE    = re.compile(r'\s+')

CONTRACTIONS = {
    "can't": 'cannot', "won't": 'will not', "n't": ' not',
    "'re": ' are', "'s": ' is', "'m": ' am', "'ll": ' will',
    "'ve": ' have', "'d": ' would',
}

def clean_text(text):
    text = str(text).strip().lower()
    for k, v in CONTRACTIONS.items():
        text = text.replace(k, v)
    text = URL_RE.sub(' url ', text)
    text = MENTION_RE.sub(' user ', text)
    text = HASHTAG_RE.sub(r' \1 ', text)
    text = EMOJI_RE.sub(' ', text)
    text = REPEAT_RE.sub(r'\1\1', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return SPACE_RE.sub(' ', text).strip()

# Apply cleaning
for split_df in [df_train, df_val, df_test, df_all]:
    split_df['clean_text'] = split_df['text'].apply(clean_text)
    split_df['char_len_clean'] = split_df['clean_text'].str.len()

# Show before/after comparison
print('Text Cleaning Examples:')
print('-' * 80)
for _, row in df_train.sample(5, random_state=SEED).iterrows():
    print(f"RAW  : {row['text'][:100]}")
    print(f"CLEAN: {row['clean_text'][:100]}")
    print()

# Before/After length plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Text Length: Before vs After Cleaning', fontsize=14, color='white')
for ax, col, title, color in zip(
    axes,
    ['char_len', 'char_len_clean'],
    ['Raw Text Length', 'Cleaned Text Length'],
    ['#3b82f6', '#10b981']
):
    ax.hist(df_train[col], bins=50, color=color, edgecolor='none', alpha=0.85)
    ax.axvline(df_train[col].mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean: {df_train[col].mean():.0f}')
    ax.set_title(title, color='white')
    ax.set_xlabel('Character Count'); ax.set_ylabel('Frequency')
    ax.legend(facecolor='#1f2937', labelcolor='white')
    ax.set_facecolor('#111827')

fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

reduction = (1 - df_train['char_len_clean'].mean() / df_train['char_len'].mean()) * 100
print(f'Length reduction: {reduction:.1f}%')

In [ ]:
# ── TF-IDF Vectorization ─────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(df_train['clean_text'])
X_val_tfidf   = tfidf.transform(df_val['clean_text'])
X_test_tfidf  = tfidf.transform(df_test['clean_text'])

y_train = df_train['label'].values
y_val   = df_val['label'].values
y_test  = df_test['label'].values

print(f'TF-IDF matrix shape (train): {X_train_tfidf.shape}')
print(f'TF-IDF matrix shape (test) : {X_test_tfidf.shape}')

# Top TF-IDF features visualization
feature_names = tfidf.get_feature_names_out()
mean_tfidf = np.array(X_train_tfidf.mean(axis=0)).flatten()
top_idx = mean_tfidf.argsort()[-30:][::-1]
top_features = [(feature_names[i], mean_tfidf[i]) for i in top_idx]

fig, ax = plt.subplots(figsize=(14, 6))
words, scores = zip(*top_features)
bars = ax.bar(range(len(words)), scores, color='#6366f1', alpha=0.85, edgecolor='none')
ax.set_xticks(range(len(words)))
ax.set_xticklabels(words, rotation=45, ha='right', fontsize=9)
ax.set_title('Top 30 TF-IDF Features (by mean score)', color='white', fontsize=14)
ax.set_ylabel('Mean TF-IDF Score')
ax.set_facecolor('#111827')
fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

In [ ]:
# ── TF-IDF discriminative features: top words per class ─────────────────
TOP_N = 15
per_class_tfidf = {}
for label_idx, label in enumerate(LABEL_NAMES):
    mask = y_train == label_idx
    mean_score = np.array(X_train_tfidf[mask].mean(axis=0)).flatten()
    top_idx = mean_score.argsort()[-TOP_N:][::-1]
    per_class_tfidf[label] = [(feature_names[i], mean_score[i]) for i in top_idx]

# Collect union of top words
all_top_words = []
for words_scores in per_class_tfidf.values():
    all_top_words.extend([w for w, _ in words_scores])
all_top_words = list(dict.fromkeys(all_top_words))  # unique, preserve order

# Build matrix: classes × words
heat_data = {}
for label in LABEL_NAMES:
    score_dict = dict(per_class_tfidf[label])
    heat_data[label] = [score_dict.get(w, 0.0) for w in all_top_words]

heat_df = pd.DataFrame(heat_data, index=all_top_words)

# Plotly heatmap
fig = go.Figure(go.Heatmap(
    z=heat_df.values.T,
    x=all_top_words, y=[l.capitalize() for l in LABEL_NAMES],
    colorscale='Viridis', showscale=True,
    hoverongaps=False,
    colorbar=dict(title='Mean TF-IDF')
))
fig.update_layout(
    template=PLOTLY_TEMPLATE, height=350,
    title='TF-IDF Feature Importance Heatmap — Top Words per Class',
    xaxis=dict(tickangle=-40, tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=12))
)
fig.show()

# Seaborn version (publication-quality static)
fig2, ax = plt.subplots(figsize=(18, 4))
sns.heatmap(heat_df.T, cmap='YlOrRd', ax=ax, linewidths=0.4, linecolor='#374151',
            cbar_kws={'label': 'Mean TF-IDF'})
ax.set_title('TF-IDF Discriminative Features per Sentiment Class', color='white', fontsize=13)
ax.set_xlabel('Word / Bigram'); ax.set_ylabel('Class')
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.set_facecolor('#111827')
fig2.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()


In [ ]:
# ── PCA + t-SNE 2-D projection of TF-IDF feature space ─────────────────
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE

SAMPLE_N = 3000   # subsample for speed
rng = np.random.RandomState(SEED)
idx = rng.choice(len(y_train), size=min(SAMPLE_N, len(y_train)), replace=False)
X_sub = X_train_tfidf[idx]
y_sub = y_train[idx]
label_sub = [LABEL_NAMES[l] for l in y_sub]

# Step 1: reduce to 50 dims with TruncatedSVD (fast on sparse)
print('Running TruncatedSVD → 50 dims...')
svd = TruncatedSVD(n_components=50, random_state=SEED)
X_svd = svd.fit_transform(X_sub)
print(f'Explained variance (50 comps): {svd.explained_variance_ratio_.sum()*100:.1f}%')

# Step 2: PCA scatter (fast)
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_svd)
pca_df = pd.DataFrame({'PC1': X_pca[:,0], 'PC2': X_pca[:,1], 'Sentiment': label_sub})

fig_pca = px.scatter(
    pca_df, x='PC1', y='PC2', color='Sentiment',
    color_discrete_map=PLOTLY_COLORS,
    opacity=0.55, template=PLOTLY_TEMPLATE,
    title=f'PCA — TF-IDF Feature Space ({SAMPLE_N} train samples)',
    labels={'Sentiment': 'True Label'},
    hover_data={'PC1': ':.3f', 'PC2': ':.3f'}
)
fig_pca.update_traces(marker=dict(size=4))
fig_pca.update_layout(height=480, legend=dict(title='Sentiment'))
fig_pca.show()

# Step 3: t-SNE (on the 50-dim SVD output)
print('Running t-SNE (perplexity=40)... this may take ~60 s')
tsne = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=SEED, verbose=0)
X_tsne = tsne.fit_transform(X_svd)
tsne_df = pd.DataFrame({'Dim1': X_tsne[:,0], 'Dim2': X_tsne[:,1], 'Sentiment': label_sub})

fig_tsne = px.scatter(
    tsne_df, x='Dim1', y='Dim2', color='Sentiment',
    color_discrete_map=PLOTLY_COLORS,
    opacity=0.55, template=PLOTLY_TEMPLATE,
    title=f't-SNE — TF-IDF Feature Space ({SAMPLE_N} train samples)',
    labels={'Sentiment': 'True Label'}
)
fig_tsne.update_traces(marker=dict(size=4))
fig_tsne.update_layout(height=520, legend=dict(title='Sentiment'))
fig_tsne.show()
print('t-SNE complete.')


In [ ]:
# ── Keras Tokenizer for RNN ──────────────────────────────────────────────────
VOCAB_SIZE  = 25_000
MAX_LEN     = 60
OOV_TOKEN   = '<OOV>'

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(df_train['clean_text'])

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(df_train['clean_text']),
                             maxlen=MAX_LEN, padding='post', truncating='post')
X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(df_val['clean_text']),
                             maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(df_test['clean_text']),
                             maxlen=MAX_LEN, padding='post', truncating='post')

# One-hot encode labels for keras
y_train_cat = keras.utils.to_categorical(y_train, num_classes=3)
y_val_cat   = keras.utils.to_categorical(y_val,   num_classes=3)
y_test_cat  = keras.utils.to_categorical(y_test,  num_classes=3)

print(f'Vocabulary size    : {len(tokenizer.word_index):,}')
print(f'Padded seq shape   : {X_train_seq.shape}')

# Sequence length distribution
seq_lens = [len(tokenizer.texts_to_sequences([t])[0]) for t in df_train['clean_text'][:5000]]
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(seq_lens, bins=50, color='#8b5cf6', alpha=0.85, edgecolor='none')
ax.axvline(MAX_LEN, color='red', linestyle='--', linewidth=2, label=f'MAX_LEN={MAX_LEN}')
ax.axvline(np.mean(seq_lens), color='#10b981', linestyle='--', linewidth=2,
           label=f'Mean={np.mean(seq_lens):.0f}')
ax.set_title('Token Sequence Length Distribution (first 5K train samples)', color='white')
ax.set_xlabel('Tokens'); ax.set_ylabel('Frequency')
ax.legend(facecolor='#1f2937', labelcolor='white')
ax.set_facecolor('#111827')
fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

coverage = np.mean([l <= MAX_LEN for l in seq_lens]) * 100
print(f'\nSequences within MAX_LEN={MAX_LEN}: {coverage:.1f}%')

---
## 🤖 Phase 3 — Classical ML Models

VADER baseline + Logistic Regression + Random Forest + SVM

In [ ]:
# ── VADER Baseline ───────────────────────────────────────────────────────────
vader = SentimentIntensityAnalyzer()

def vader_predict(text):
    scores = vader.polarity_scores(str(text))
    compound = scores['compound']
    if compound >= 0.05:
        return 2  # positive
    elif compound <= -0.05:
        return 0  # negative
    else:
        return 1  # neutral

def vader_proba(text):
    scores = vader.polarity_scores(str(text))
    pos, neg, neu = scores['pos'], scores['neg'], scores['neu']
    total = pos + neg + neu or 1
    return [neg/total, neu/total, pos/total]

print('Running VADER on test set...')
vader_preds = [vader_predict(t) for t in df_test['text']]
vader_probas = np.array([vader_proba(t) for t in df_test['text']])

vader_f1   = f1_score(y_test, vader_preds, average='macro')
vader_prec = precision_score(y_test, vader_preds, average='macro', zero_division=0)
vader_rec  = recall_score(y_test, vader_preds, average='macro', zero_division=0)

print(f'VADER — Macro F1: {vader_f1:.4f} | Precision: {vader_prec:.4f} | Recall: {vader_rec:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, vader_preds, target_names=LABEL_NAMES))

# VADER compound score distribution
vader_scores = df_train['text'].apply(lambda t: vader.polarity_scores(str(t))['compound'])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('VADER Analysis', fontsize=14, color='white')

for label in LABEL_NAMES:
    idx = df_train['label_name'] == label
    axes[0].hist(vader_scores[idx], bins=50, alpha=0.6, label=label, color=COLORS[label])
axes[0].axvline(0.05, color='green', linestyle='--', alpha=0.5, label='pos threshold')
axes[0].axvline(-0.05, color='red', linestyle='--', alpha=0.5, label='neg threshold')
axes[0].set_title('VADER Compound Score Distribution', color='white')
axes[0].set_xlabel('Compound Score'); axes[0].set_ylabel('Frequency')
axes[0].legend(facecolor='#1f2937', labelcolor='white')
axes[0].set_facecolor('#111827')

cm = confusion_matrix(y_test, vader_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            linewidths=0.5, linecolor='#374151')
axes[1].set_title('VADER Confusion Matrix', color='white')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].set_facecolor('#111827')
fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

In [ ]:
# ── Logistic Regression ──────────────────────────────────────────────────────
print('Training Logistic Regression (GridSearchCV)...')
param_grid = {'C': [0.01, 0.1, 1.0, 10.0]}
lr_cv = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=SEED, multi_class='multinomial', solver='lbfgs'),
    param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=0
)
lr_cv.fit(X_train_tfidf, y_train)
lr_model = lr_cv.best_estimator_

print(f'Best C: {lr_cv.best_params_["C"]}  |  CV F1: {lr_cv.best_score_:.4f}')

lr_preds  = lr_model.predict(X_test_tfidf)
lr_probas = lr_model.predict_proba(X_test_tfidf)
lr_f1     = f1_score(y_test, lr_preds, average='macro')
lr_prec   = precision_score(y_test, lr_preds, average='macro', zero_division=0)
lr_rec    = recall_score(y_test, lr_preds, average='macro', zero_division=0)

print(f'LR — Macro F1: {lr_f1:.4f} | Precision: {lr_prec:.4f} | Recall: {lr_rec:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, lr_preds, target_names=LABEL_NAMES))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, lr_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            linewidths=0.5, linecolor='#374151')
ax.set_title('Logistic Regression — Confusion Matrix', color='white', fontsize=13)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_facecolor('#111827')
fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

In [ ]:
# ── Random Forest ────────────────────────────────────────────────────────────
print('Training Random Forest (300 trees)...')
rf_model = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1,
                                   max_depth=None, min_samples_split=5)
rf_model.fit(X_train_tfidf, y_train)

rf_preds  = rf_model.predict(X_test_tfidf)
rf_probas = rf_model.predict_proba(X_test_tfidf)
rf_f1     = f1_score(y_test, rf_preds, average='macro')
rf_prec   = precision_score(y_test, rf_preds, average='macro', zero_division=0)
rf_rec    = recall_score(y_test, rf_preds, average='macro', zero_division=0)

print(f'RF — Macro F1: {rf_f1:.4f} | Precision: {rf_prec:.4f} | Recall: {rf_rec:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, rf_preds, target_names=LABEL_NAMES))

# Feature importance
importances = rf_model.feature_importances_
top_rf_idx = importances.argsort()[-25:][::-1]
top_rf_features = feature_names[top_rf_idx]
top_rf_scores = importances[top_rf_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# Confusion matrix
cm = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, linewidths=0.5)
axes[0].set_title('Random Forest — Confusion Matrix', color='white')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
# Feature importance
axes[1].barh(range(25), top_rf_scores[::-1], color='#f97316', alpha=0.85, edgecolor='none')
axes[1].set_yticks(range(25)); axes[1].set_yticklabels(top_rf_features[::-1], fontsize=9)
axes[1].set_title('Random Forest — Top 25 Feature Importances', color='white')
axes[1].set_xlabel('Importance Score')

for ax in axes: ax.set_facecolor('#111827')
fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

In [ ]:
# ── SVM (LinearSVC + Calibration) ───────────────────────────────────────────
print('Training LinearSVC with probability calibration...')
svm_base  = LinearSVC(max_iter=2000, random_state=SEED, C=1.0)
svm_model = CalibratedClassifierCV(svm_base, cv=3)
svm_model.fit(X_train_tfidf, y_train)

svm_preds  = svm_model.predict(X_test_tfidf)
svm_probas = svm_model.predict_proba(X_test_tfidf)
svm_f1     = f1_score(y_test, svm_preds, average='macro')
svm_prec   = precision_score(y_test, svm_preds, average='macro', zero_division=0)
svm_rec    = recall_score(y_test, svm_preds, average='macro', zero_division=0)

print(f'SVM — Macro F1: {svm_f1:.4f} | Precision: {svm_prec:.4f} | Recall: {svm_rec:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, svm_preds, target_names=LABEL_NAMES))

# ROC curves for SVM
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cm = confusion_matrix(y_test, svm_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, linewidths=0.5)
axes[0].set_title('SVM — Confusion Matrix', color='white')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

for i, label in enumerate(LABEL_NAMES):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], svm_probas[:, i])
    auc = roc_auc_score(y_test_bin[:, i], svm_probas[:, i])
    axes[1].plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})', color=list(COLORS.values())[i], linewidth=2)
axes[1].plot([0,1],[0,1],'--', color='gray', alpha=0.5)
axes[1].set_title('SVM — ROC Curves (One-vs-Rest)', color='white')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(facecolor='#1f2937', labelcolor='white')

for ax in axes: ax.set_facecolor('#111827')
fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

In [ ]:
# ── Confidence score distributions per model (Plotly violin) ─────────────
# Build a long-form dataframe: model × true_class × predicted_confidence
conf_rows = []
model_info = [
    ('VADER',               vader_probas),
    ('Logistic Regression', lr_probas),
    ('Random Forest',       rf_probas),
    ('SVM',                 svm_probas),
]
for model_name, probas in model_info:
    for i, (prob_row, true_lbl) in enumerate(zip(probas, y_test)):
        pred_lbl = prob_row.argmax()
        conf_rows.append({
            'Model':       model_name,
            'True Class':  LABEL_NAMES[true_lbl],
            'Pred Class':  LABEL_NAMES[pred_lbl],
            'Confidence':  float(prob_row.max()),
            'Correct':     pred_lbl == true_lbl,
        })

bilstm_preds_arr = bilstm_probas.argmax(axis=1)
for i, (prob_row, true_lbl) in enumerate(zip(bilstm_probas, y_test)):
    conf_rows.append({
        'Model': 'BiLSTM', 'True Class': LABEL_NAMES[true_lbl],
        'Pred Class': LABEL_NAMES[prob_row.argmax()],
        'Confidence': float(prob_row.max()),
        'Correct': prob_row.argmax() == true_lbl,
    })

conf_df = pd.DataFrame(conf_rows)

# Violin plot — confidence by model, coloured by correct/wrong
fig = px.violin(
    conf_df, x='Model', y='Confidence', color='Correct',
    color_discrete_map={True: '#10b981', False: '#ef4444'},
    box=True, points=False,
    template=PLOTLY_TEMPLATE, height=480,
    title='Confidence Score Distributions — Correct vs Incorrect Predictions',
    labels={'Correct': 'Prediction', 'Confidence': 'Max Class Probability'},
    category_orders={'Correct': [True, False]}
)
fig.update_traces(meanline_visible=True)
fig.for_each_trace(lambda t: t.update(name='Correct' if t.name=='True' else 'Incorrect'))
fig.show()

# Per-true-class breakdown
fig2 = px.box(
    conf_df, x='True Class', y='Confidence', color='Model',
    template=PLOTLY_TEMPLATE, height=440,
    title='Confidence by True Sentiment Class and Model',
    category_orders={'True Class': LABEL_NAMES}
)
fig2.show()

# Summary table
summary = conf_df.groupby(['Model','Correct'])['Confidence'].agg(['mean','median','std']).round(3)
print(summary.to_string())


---
## 🧠 Phase 4 — Bidirectional LSTM with Attention

Deep learning approach: `Embedding → BiLSTM → Attention → Dense(3)`

In [ ]:
# ── Build BiLSTM model with custom Attention ─────────────────────────────────
EMBED_DIM   = 128
LSTM_UNITS  = 128
DROPOUT     = 0.3
BATCH_SIZE  = 64
EPOCHS      = 15

class BahdanauAttention(layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, encoder_output):
        score = self.V(tf.nn.tanh(self.W(encoder_output)))  # (batch, seq, 1)
        weights = tf.nn.softmax(score, axis=1)               # (batch, seq, 1)
        context = weights * encoder_output                   # (batch, seq, units)
        context = tf.reduce_sum(context, axis=1)             # (batch, units)
        return context, tf.squeeze(weights, -1)


def build_bilstm():
    inp = keras.Input(shape=(MAX_LEN,), name='input')
    x   = layers.Embedding(VOCAB_SIZE, EMBED_DIM, mask_zero=True, name='embedding')(inp)
    x   = layers.SpatialDropout1D(DROPOUT)(x)
    x   = layers.Bidirectional(layers.LSTM(LSTM_UNITS, return_sequences=True,
                                            dropout=DROPOUT, recurrent_dropout=0.1),
                                name='bilstm')(x)
    ctx, weights = BahdanauAttention(64, name='attention')(x)
    x   = layers.Dense(64, activation='relu')(ctx)
    x   = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(3, activation='softmax', name='output')(x)
    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


bilstm_model = build_bilstm()
bilstm_model.summary()

In [ ]:
# ── Train BiLSTM ─────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1),
]

print('Training BiLSTM...')
history = bilstm_model.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_val_seq, y_val_cat),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=callbacks, verbose=1
)
print('Training complete ✓')

In [ ]:
# ── Training history plots ────────────────────────────────────────────────────
hist = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('BiLSTM Training History', fontsize=14, color='white')

# Accuracy
axes[0].plot(epochs_ran, hist['accuracy'],     color='#6366f1', linewidth=2.5, label='Train Acc')
axes[0].plot(epochs_ran, hist['val_accuracy'], color='#10b981', linewidth=2.5,
             linestyle='--', label='Val Acc')
axes[0].set_title('Accuracy', color='white')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(facecolor='#1f2937', labelcolor='white')
axes[0].set_facecolor('#111827')

# Loss
axes[1].plot(epochs_ran, hist['loss'],     color='#f97316', linewidth=2.5, label='Train Loss')
axes[1].plot(epochs_ran, hist['val_loss'], color='#ef4444', linewidth=2.5,
             linestyle='--', label='Val Loss')
axes[1].set_title('Loss', color='white')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(facecolor='#1f2937', labelcolor='white')
axes[1].set_facecolor('#111827')

fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

print(f"Best val_accuracy : {max(hist['val_accuracy']):.4f}")
print(f"Best val_loss     : {min(hist['val_loss']):.4f}")

In [ ]:
# ── BiLSTM Evaluation ────────────────────────────────────────────────────────
bilstm_probas = bilstm_model.predict(X_test_seq, verbose=0)
bilstm_preds  = bilstm_probas.argmax(axis=1)
bilstm_f1     = f1_score(y_test, bilstm_preds, average='macro')
bilstm_prec   = precision_score(y_test, bilstm_preds, average='macro', zero_division=0)
bilstm_rec    = recall_score(y_test, bilstm_preds, average='macro', zero_division=0)

print(f'BiLSTM — Macro F1: {bilstm_f1:.4f} | Precision: {bilstm_prec:.4f} | Recall: {bilstm_rec:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, bilstm_preds, target_names=LABEL_NAMES))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# Confusion matrix
cm = confusion_matrix(y_test, bilstm_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, linewidths=0.5)
axes[0].set_title('BiLSTM — Confusion Matrix', color='white')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

# ROC curves
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
for i, label in enumerate(LABEL_NAMES):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], bilstm_probas[:, i])
    auc = roc_auc_score(y_test_bin[:, i], bilstm_probas[:, i])
    axes[1].plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})',
                 color=list(COLORS.values())[i], linewidth=2)
axes[1].plot([0,1],[0,1],'--', color='gray', alpha=0.5)
axes[1].set_title('BiLSTM — ROC Curves (One-vs-Rest)', color='white')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(facecolor='#1f2937', labelcolor='white')

for ax in axes: ax.set_facecolor('#111827')
fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

In [ ]:
# ── Calibration curves (reliability diagrams) for all 5 models ──────────
from sklearn.calibration import calibration_curve

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

model_probas = {
    'VADER':               vader_probas,
    'Logistic Regression': lr_probas,
    'Random Forest':       rf_probas,
    'SVM':                 svm_probas,
    'BiLSTM':              bilstm_probas,
}

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Negative class','Neutral class','Positive class'],
    shared_yaxes=True
)

model_palette = {
    'VADER': '#ef4444','Logistic Regression': '#3b82f6',
    'Random Forest': '#f97316','SVM': '#8b5cf6','BiLSTM': '#10b981'
}

for class_idx, class_name in enumerate(LABEL_NAMES):
    col = class_idx + 1
    # Perfect calibration reference
    fig.add_trace(
        go.Scatter(x=[0,1], y=[0,1], mode='lines',
                   line=dict(dash='dash', color='gray', width=1),
                   showlegend=(col==1), name='Perfect calibration'),
        row=1, col=col
    )
    for model_name, probas in model_probas.items():
        try:
            frac_pos, mean_pred = calibration_curve(
                y_test_bin[:, class_idx], probas[:, class_idx], n_bins=10
            )
            fig.add_trace(
                go.Scatter(x=mean_pred, y=frac_pos, mode='lines+markers',
                           name=model_name, showlegend=(col==1),
                           line=dict(color=model_palette[model_name], width=2),
                           marker=dict(size=6)),
                row=1, col=col
            )
        except Exception:
            pass

fig.update_layout(
    template=PLOTLY_TEMPLATE, height=420,
    title_text='Calibration Curves (Reliability Diagrams) — All Models',
    title_font_size=15,
    legend=dict(x=1.02, y=1)
)
fig.update_xaxes(title_text='Mean Predicted Probability', range=[0,1])
fig.update_yaxes(title_text='Fraction of Positives', range=[0,1], col=1)
fig.show()


In [ ]:
# ── Attention Heatmap Visualization ─────────────────────────────────────────
# Build a model that also returns attention weights
attention_model = keras.Model(
    inputs=bilstm_model.input,
    outputs=[
        bilstm_model.output,
        bilstm_model.get_layer('attention').output[1]  # attention weights
    ]
)

def get_attention(text):
    seq = pad_sequences(tokenizer.texts_to_sequences([clean_text(text)]),
                        maxlen=MAX_LEN, padding='post')
    probs, attn = attention_model.predict(seq, verbose=0)
    words = clean_text(text).split()
    weights = attn[0, :len(words)]
    return words, weights, probs[0]

EXAMPLES = [
    'I absolutely love this product! Best purchase ever!',
    'The service was terrible and the staff were rude.',
    'Just received my order. Packaging was fine.',
]

fig, axes = plt.subplots(len(EXAMPLES), 1, figsize=(16, 5 * len(EXAMPLES)))
fig.suptitle('BiLSTM Attention Heatmaps — Token Importance', fontsize=14, color='white')

for ax, example in zip(axes, EXAMPLES):
    words, weights, probs = get_attention(example)
    pred_label = LABEL_NAMES[probs.argmax()]

    if len(words) == 0:
        ax.set_visible(False)
        continue

    norm_weights = (weights - weights.min()) / (weights.max() - weights.min() + 1e-8)
    colors = plt.cm.YlOrRd(norm_weights)

    bars = ax.bar(range(len(words)), norm_weights, color=colors, width=0.8, edgecolor='none')
    ax.set_xticks(range(len(words)))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel('Attention Weight')
    ax.set_title(f'"{example[:60]}..."  →  {pred_label.upper()}  (conf: {probs.max()*100:.1f}%)',
                 color=COLORS.get(pred_label, 'white'), fontsize=11)
    ax.set_facecolor('#111827')

fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

---
## 📊 Phase 5 — All Models Comparison & Benchmarking

Comprehensive comparison across all 5 models.

In [ ]:
# ── Build benchmark DataFrame ─────────────────────────────────────────────────
benchmark = pd.DataFrame([
    {'model': 'VADER',               'macro_f1': vader_f1,   'macro_precision': vader_prec,  'macro_recall': vader_rec,  'type': 'Rule-based'},
    {'model': 'Logistic Regression', 'macro_f1': lr_f1,      'macro_precision': lr_prec,     'macro_recall': lr_rec,     'type': 'Classical ML'},
    {'model': 'Random Forest',       'macro_f1': rf_f1,      'macro_precision': rf_prec,     'macro_recall': rf_rec,     'type': 'Classical ML'},
    {'model': 'SVM',                 'macro_f1': svm_f1,     'macro_precision': svm_prec,    'macro_recall': svm_rec,    'type': 'Classical ML'},
    {'model': 'BiLSTM',              'macro_f1': bilstm_f1,  'macro_precision': bilstm_prec, 'macro_recall': bilstm_rec, 'type': 'Deep Learning'},
]).sort_values('macro_f1', ascending=False).reset_index(drop=True)

benchmark['rank'] = range(1, len(benchmark)+1)
print('=' * 70)
print('ALL MODELS BENCHMARK — tweet_eval/sentiment test split')
print('=' * 70)
print(benchmark[['rank','model','type','macro_f1','macro_precision','macro_recall']].to_string(index=False))
print('=' * 70)

best = benchmark.iloc[0]
vader_row = benchmark[benchmark['model']=='VADER'].iloc[0]
print(f'\nBest model: {best["model"]} (F1={best["macro_f1"]:.4f})')
print(f'Improvement over VADER: +{best["macro_f1"]-vader_row["macro_f1"]:.4f} F1')

In [ ]:
# ── Comprehensive comparison charts ──────────────────────────────────────────
model_names = benchmark['model'].tolist()
colors_by_model = {
    'VADER': '#ef4444', 'Logistic Regression': '#3b82f6',
    'Random Forest': '#f97316', 'SVM': '#8b5cf6', 'BiLSTM': '#10b981'
}
bar_colors = [colors_by_model[m] for m in benchmark['model']]

fig = plt.figure(figsize=(20, 16))
fig.patch.set_facecolor('#0f172a')
fig.suptitle('Model Comparison — Complete Benchmark Results', fontsize=18, color='white', y=0.98)

# 1. F1 bar chart
ax1 = fig.add_subplot(2, 3, 1)
bars = ax1.bar(benchmark['model'], benchmark['macro_f1'], color=bar_colors, alpha=0.9, width=0.6, edgecolor='none')
for bar, val in zip(bars, benchmark['macro_f1']):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{val:.3f}',
             ha='center', va='bottom', color='white', fontsize=10, fontweight='bold')
ax1.set_title('Macro F1 Score', color='white', fontsize=13)
ax1.set_ylabel('Score'); ax1.set_xticklabels(benchmark['model'], rotation=30, ha='right', fontsize=9)
ax1.set_ylim(0, 1.05); ax1.axhline(0.7, color='white', linestyle='--', alpha=0.3, linewidth=1)
ax1.set_facecolor('#111827')

# 2. Multi-metric grouped bar
ax2 = fig.add_subplot(2, 3, 2)
x = np.arange(len(model_names))
w = 0.26
ax2.bar(x-w, benchmark['macro_f1'],        width=w, color='#6366f1', alpha=0.9, label='F1',        edgecolor='none')
ax2.bar(x,   benchmark['macro_precision'],  width=w, color='#3b82f6', alpha=0.9, label='Precision', edgecolor='none')
ax2.bar(x+w, benchmark['macro_recall'],     width=w, color='#10b981', alpha=0.9, label='Recall',    edgecolor='none')
ax2.set_xticks(x); ax2.set_xticklabels(
    ['VADER','LR','RF','SVM','BiLSTM'][: len(model_names)], rotation=0, fontsize=9
)
ax2.set_title('F1 / Precision / Recall', color='white', fontsize=13)
ax2.set_ylim(0, 1.05); ax2.legend(facecolor='#1f2937', labelcolor='white')
ax2.set_facecolor('#111827')

# 3. Radar chart
ax3 = fig.add_subplot(2, 3, 3, projection='polar')
metrics = ['F1', 'Precision', 'Recall']
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]
for _, row in benchmark.iterrows():
    vals = [row['macro_f1'], row['macro_precision'], row['macro_recall']] + [row['macro_f1']]
    color = colors_by_model.get(row['model'], '#6366f1')
    ax3.plot(angles, vals, color=color, linewidth=2, label=row['model'])
    ax3.fill(angles, vals, alpha=0.05, color=color)
ax3.set_xticks(angles[:-1]); ax3.set_xticklabels(metrics, color='white', fontsize=11)
ax3.set_ylim(0, 1); ax3.set_title('Radar Chart', color='white', fontsize=13, pad=20)
ax3.set_facecolor('#111827'); ax3.spines['polar'].set_color('#374151')
ax3.grid(color='#374151')
ax3.tick_params(colors='white')
ax3.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), facecolor='#1f2937', labelcolor='white', fontsize=9)

# 4. Heatmap — per-class F1
ax4 = fig.add_subplot(2, 3, 4)
all_preds = {'VADER': vader_preds, 'Logistic Regression': lr_preds,
             'Random Forest': rf_preds, 'SVM': svm_preds, 'BiLSTM': bilstm_preds}
per_class_data = {}
for model_name, preds in all_preds.items():
    report = classification_report(y_test, preds, target_names=LABEL_NAMES, output_dict=True)
    per_class_data[model_name] = {label: report[label]['f1-score'] for label in LABEL_NAMES}

heatmap_df = pd.DataFrame(per_class_data, index=LABEL_NAMES)
sns.heatmap(heatmap_df.T, annot=True, fmt='.3f', cmap='RdYlGn',
            ax=ax4, linewidths=0.5, linecolor='#374151',
            vmin=0, vmax=1, cbar_kws={'label': 'F1 Score'})
ax4.set_title('Per-class F1 Heatmap', color='white', fontsize=13)
ax4.set_xlabel('Class'); ax4.set_ylabel('Model')
ax4.set_facecolor('#111827')

# 5. Improvement over VADER
ax5 = fig.add_subplot(2, 3, 5)
vader_baseline = benchmark[benchmark['model']=='VADER']['macro_f1'].values[0]
improvements = benchmark['macro_f1'] - vader_baseline
bar_cols = ['#10b981' if v > 0 else '#ef4444' for v in improvements]
ax5.bar(benchmark['model'], improvements, color=bar_cols, alpha=0.9, width=0.6, edgecolor='none')
ax5.axhline(0, color='white', linewidth=1, alpha=0.5)
for i, (bar_x, val) in enumerate(zip(benchmark['model'], improvements)):
    ax5.text(i, val + (0.002 if val >= 0 else -0.006), f'{val:+.3f}',
             ha='center', color='white', fontsize=10)
ax5.set_title('F1 Improvement vs VADER Baseline', color='white', fontsize=13)
ax5.set_ylabel('ΔF1'); ax5.set_xticklabels(benchmark['model'], rotation=30, ha='right', fontsize=9)
ax5.set_facecolor('#111827')

# 6. Summary table
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis('off')
table_data = [
    [f"#{row['rank']}", row['model'], row['type'], f"{row['macro_f1']:.4f}"]
    for _, row in benchmark.iterrows()
]
t = ax6.table(
    cellText=table_data, colLabels=['Rank', 'Model', 'Type', 'Macro F1'],
    cellLoc='center', loc='center'
)
t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1, 2.2)
for (r, c), cell in t.get_celld().items():
    cell.set_edgecolor('#374151')
    cell.set_facecolor('#1f2937' if r > 0 else '#374151')
    cell.set_text_props(color='white')
ax6.set_title('Benchmark Summary', color='white', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrices — all models side-by-side ─────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(24, 5))
fig.suptitle('Confusion Matrices — All Models', fontsize=14, color='white')
cmaps = ['Reds', 'Blues', 'Oranges', 'Purples', 'Greens']

for ax, (model_name, preds), cmap in zip(
    axes,
    all_preds.items(),
    cmaps
):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['neg','neu','pos'], yticklabels=['neg','neu','pos'],
                linewidths=0.5, linecolor='#374151', cbar=False)
    f1 = f1_score(y_test, preds, average='macro')
    ax.set_title(f'{model_name}\nF1={f1:.3f}', color='white', fontsize=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_facecolor('#111827')

fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

In [ ]:
# ── Precision-Recall curves for all 5 models (macro / per-class) ─────────
from sklearn.metrics import precision_recall_curve, average_precision_score

model_probas_dict = {
    'VADER': vader_probas, 'Logistic Regression': lr_probas,
    'Random Forest': rf_probas, 'SVM': svm_probas, 'BiLSTM': bilstm_probas,
}
model_preds_dict = {
    'VADER': vader_preds, 'Logistic Regression': lr_preds,
    'Random Forest': rf_preds, 'SVM': svm_preds, 'BiLSTM': bilstm_preds,
}
y_test_bin3 = label_binarize(y_test, classes=[0, 1, 2])

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'{n} class' for n in ['Negative','Neutral','Positive']]
)
palette = {'VADER':'#ef4444','Logistic Regression':'#3b82f6',
           'Random Forest':'#f97316','SVM':'#8b5cf6','BiLSTM':'#10b981'}

for ci, class_name in enumerate(LABEL_NAMES):
    col = ci + 1
    for model_name, probas in model_probas_dict.items():
        try:
            prec, rec, _ = precision_recall_curve(y_test_bin3[:, ci], probas[:, ci])
            ap = average_precision_score(y_test_bin3[:, ci], probas[:, ci])
            fig.add_trace(
                go.Scatter(x=rec, y=prec, mode='lines',
                           name=f'{model_name} (AP={ap:.3f})',
                           showlegend=(col == 1),
                           line=dict(color=palette[model_name], width=2.5)),
                row=1, col=col
            )
        except Exception:
            pass

fig.update_layout(
    template=PLOTLY_TEMPLATE, height=440,
    title_text='Precision-Recall Curves — All Models (One-vs-Rest)',
    title_font_size=15,
    legend=dict(x=1.01, y=1, font=dict(size=10))
)
fig.update_xaxes(title_text='Recall', range=[0,1])
fig.update_yaxes(title_text='Precision', range=[0,1], col=1)
fig.show()

# Table: Average Precision per model per class
ap_rows = []
for model_name, probas in model_probas_dict.items():
    row = {'Model': model_name}
    for ci, cn in enumerate(LABEL_NAMES):
        try:
            row[f'AP ({cn})'] = round(average_precision_score(y_test_bin3[:, ci], probas[:, ci]), 4)
        except:
            row[f'AP ({cn})'] = float('nan')
    ap_rows.append(row)

ap_df = pd.DataFrame(ap_rows).set_index('Model')
print('Average Precision (AP) per class:')
display(ap_df.style
    .background_gradient(cmap='RdYlGn', vmin=0, vmax=1)
    .format('{:.4f}')
    .set_caption('Average Precision — higher is better'))


In [ ]:
# ── Plotly interactive benchmark dashboard ───────────────────────────────

# 1. Animated grouped bar
bm_long = benchmark.melt(
    id_vars=['model','type'], value_vars=['macro_f1','macro_precision','macro_recall'],
    var_name='Metric', value_name='Score'
)
bm_long['Metric'] = bm_long['Metric'].str.replace('macro_','').str.capitalize()

fig1 = px.bar(
    bm_long.sort_values('Score', ascending=False),
    x='model', y='Score', color='Metric', barmode='group',
    text='Score', template=PLOTLY_TEMPLATE,
    color_discrete_sequence=['#6366f1','#3b82f6','#10b981'],
    title='Interactive Benchmark — F1 / Precision / Recall per Model',
    labels={'model':'Model','Score':'Macro Score'},
    height=460
)
fig1.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig1.update_yaxes(range=[0, 1.05])
fig1.show()

# 2. Parallel coordinates — model quality landscape
bm_pc = benchmark.copy()
bm_pc['model_idx'] = range(len(bm_pc))
fig2 = go.Figure(go.Parcoords(
    line=dict(color=bm_pc['macro_f1'], colorscale='Viridis', showscale=True,
              colorbar=dict(title='Macro F1')),
    dimensions=[
        dict(label='Model', values=bm_pc['model_idx'],
             tickvals=list(range(len(bm_pc))),
             ticktext=bm_pc['model'].tolist()),
        dict(label='Macro F1',        values=bm_pc['macro_f1'],        range=[0,1]),
        dict(label='Macro Precision', values=bm_pc['macro_precision'],  range=[0,1]),
        dict(label='Macro Recall',    values=bm_pc['macro_recall'],     range=[0,1]),
    ]
))
fig2.update_layout(template=PLOTLY_TEMPLATE, height=380,
                   title='Parallel Coordinates — Model Quality Landscape')
fig2.show()

# 3. Sunburst — prediction breakdown by model and true class
sun_rows = []
for model_name, preds in [('VADER',vader_preds),('LR',lr_preds),
                            ('RF',rf_preds),('SVM',svm_preds),('BiLSTM',bilstm_preds)]:
    for true, pred in zip(y_test, preds):
        sun_rows.append({'Model': model_name,
                         'True': LABEL_NAMES[true], 'Predicted': LABEL_NAMES[pred]})
sun_df = pd.DataFrame(sun_rows)
sun_agg = sun_df.groupby(['Model','True','Predicted']).size().reset_index(name='Count')
sun_agg['Correct'] = (sun_agg['True'] == sun_agg['Predicted']).map({True:'Correct', False:'Wrong'})

fig3 = px.sunburst(
    sun_agg, path=['Model','True','Correct'], values='Count',
    color='Correct', color_discrete_map={'Correct':'#10b981','Wrong':'#ef4444'},
    template=PLOTLY_TEMPLATE, height=560,
    title='Prediction Sunburst — Model → True Class → Correct / Wrong'
)
fig3.show()


In [ ]:
# ── Error analysis — what gets misclassified? ────────────────────────────
BEST_MODEL_PREDS = bilstm_preds   # use best model
BEST_MODEL_PROBAS = bilstm_probas

err_df = df_test[['text','label_name','clean_text','word_count','char_len']].copy().reset_index(drop=True)
err_df['predicted']   = [LABEL_NAMES[p] for p in BEST_MODEL_PREDS]
err_df['confidence']  = BEST_MODEL_PROBAS.max(axis=1)
err_df['correct']     = err_df['label_name'] == err_df['predicted']
err_df['error_type']  = err_df.apply(
    lambda r: None if r['correct'] else f"{r['label_name']}→{r['predicted']}", axis=1
)

# High-confidence errors
high_conf_errors = err_df[(~err_df['correct']) & (err_df['confidence'] > 0.85)]
print(f'High-confidence errors (conf > 85%): {len(high_conf_errors)}')
print(f'Accuracy (BiLSTM): {err_df["correct"].mean()*100:.2f}%')

# Error type distribution
error_counts = err_df[~err_df['correct']]['error_type'].value_counts().reset_index()
error_counts.columns = ['Error Type', 'Count']

fig1 = px.bar(error_counts, x='Error Type', y='Count',
              color='Count', color_continuous_scale='Reds',
              template=PLOTLY_TEMPLATE, height=380,
              title='BiLSTM Misclassification Pattern (True→Predicted)')
fig1.show()

# Word count vs confidence (correct vs wrong)
sample = err_df.sample(min(3000, len(err_df)), random_state=SEED)
fig2 = px.scatter(
    sample, x='word_count', y='confidence', color='correct',
    color_discrete_map={True:'#10b981', False:'#ef4444'},
    opacity=0.5, template=PLOTLY_TEMPLATE, height=430,
    title='Word Count vs Confidence — Correct vs Incorrect (BiLSTM)',
    labels={'word_count':'Word Count','confidence':'Max Confidence','correct':'Correct?'},
    hover_data={'text': True}
)
fig2.show()

# Show worst high-confidence errors
print('\nTop high-confidence misclassifications (BiLSTM):')
display(high_conf_errors[['text','label_name','predicted','confidence']]
        .sort_values('confidence', ascending=False)
        .head(10)
        .style
        .background_gradient(subset=['confidence'], cmap='Reds')
        .format({'confidence': '{:.3f}'})
        .set_caption('High-confidence errors — model was very wrong'))


---
## ⚡ Phase 6 — Live Inference Demo

Run all 5 models on custom text inputs and display results with attention visualization.

In [ ]:
# ── Inference pipeline for all models ───────────────────────────────────────
def predict_all(text):
    """Run all 5 models on text. Returns a results dict."""
    clean = clean_text(text)

    # VADER
    v_label = vader_predict(text)
    v_proba = vader_proba(text)

    # TF-IDF features
    vec = tfidf.transform([clean])

    # LR
    lr_p = lr_model.predict_proba(vec)[0]
    lr_l = lr_model.predict(vec)[0]

    # RF
    rf_p = rf_model.predict_proba(vec)[0]
    rf_l = rf_model.predict(vec)[0]

    # SVM
    sv_p = svm_model.predict_proba(vec)[0]
    sv_l = svm_model.predict(vec)[0]

    # BiLSTM + Attention
    seq = pad_sequences(tokenizer.texts_to_sequences([clean]),
                        maxlen=MAX_LEN, padding='post')
    bl_p, attn = attention_model.predict(seq, verbose=0)
    bl_p = bl_p[0]; bl_l = bl_p.argmax()
    tokens = clean.split()
    weights = attn[0, :len(tokens)]

    return {
        'text': text,
        'vader':  {'label': LABEL_NAMES[v_label], 'confidence': max(v_proba), 'proba': v_proba},
        'lr':     {'label': LABEL_NAMES[lr_l],    'confidence': max(lr_p),    'proba': lr_p.tolist()},
        'rf':     {'label': LABEL_NAMES[rf_l],    'confidence': max(rf_p),    'proba': rf_p.tolist()},
        'svm':    {'label': LABEL_NAMES[sv_l],    'confidence': max(sv_p),    'proba': sv_p.tolist()},
        'bilstm': {'label': LABEL_NAMES[bl_l],    'confidence': float(bl_p.max()), 'proba': bl_p.tolist()},
        'attention': {'tokens': tokens, 'weights': weights.tolist()},
    }


def display_results(results):
    """Display inference results as a table with attention heatmap."""
    print(f'\nText: "{results["text"]}"')
    print('=' * 70)

    rows = []
    for model in ['vader','lr','rf','svm','bilstm']:
        r = results[model]
        rows.append({
            'Model': model.upper(), 'Sentiment': r['label'].capitalize(),
            'Confidence': f"{r['confidence']*100:.1f}%",
            'Negative': f"{r['proba'][0]*100:.1f}%",
            'Neutral':  f"{r['proba'][1]*100:.1f}%",
            'Positive': f"{r['proba'][2]*100:.1f}%",
        })

    display_df = pd.DataFrame(rows)
    print(display_df.to_string(index=False))

    # Consensus
    labels = [results[m]['label'] for m in ['vader','lr','rf','svm','bilstm']]
    vote = Counter(labels).most_common(1)[0]
    print(f'\n🗳️  Consensus: {vote[0].upper()} ({vote[1]}/5 models agree)')

    # Attention heatmap
    if results.get('attention') and results['attention']['tokens']:
        tokens  = results['attention']['tokens']
        weights = np.array(results['attention']['weights'])
        if len(weights) > 0:
            norm_w = (weights - weights.min()) / (weights.max() - weights.min() + 1e-8)
            fig, ax = plt.subplots(figsize=(min(18, max(8, len(tokens)*0.8)), 2.5))
            colors = plt.cm.YlOrRd(norm_w)
            ax.bar(range(len(tokens)), norm_w, color=colors, width=0.8, edgecolor='none')
            ax.set_xticks(range(len(tokens)))
            ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=11)
            ax.set_title(f'BiLSTM Attention Heatmap — "{results["text"][:50]}..."', color='white')
            ax.set_ylabel('Attention Weight')
            ax.set_facecolor('#111827')
            fig.patch.set_facecolor('#0f172a')
            plt.tight_layout()
            plt.show()

print('Inference pipeline ready ✓')

In [ ]:
# ── Run live inference on demo texts ─────────────────────────────────────────
DEMO_TEXTS = [
    'I absolutely love this product! Best purchase ever! Totally recommend it to everyone!',
    'The service was terrible and the staff were extremely rude. Worst experience of my life.',
    'Just received my order. Packaging was fine. Arrived on time.',
    'This is going to change everything we know about machine learning forever!',
    'Not bad, but the price is a bit high for what you get.',
]

for text in DEMO_TEXTS:
    results = predict_all(text)
    display_results(results)
    print('\n' + '-'*70 + '\n')

In [ ]:
# ── Custom inference — change this text ─────────────────────────────────────
YOUR_TEXT = "Type any social media post or tweet here to analyze its sentiment!"

results = predict_all(YOUR_TEXT)
display_results(results)

---
## 📋 Final Summary

Complete benchmark results and key findings from the project.

In [ ]:
# ── Final benchmark table ────────────────────────────────────────────────────
print('=' * 70)
print('FINAL BENCHMARK — Social Media Sentiment Analysis')
print('Dataset: tweet_eval/sentiment | Test split | Macro-averaged metrics')
print('=' * 70)
print(f'{"Rank":<5} {"Model":<22} {"Type":<15} {"F1":>8} {"Prec":>8} {"Recall":>8}')
print('-' * 70)
for _, row in benchmark.iterrows():
    print(f"{int(row['rank']):<5} {row['model']:<22} {row['type']:<15} "
          f"{row['macro_f1']:>8.4f} {row['macro_precision']:>8.4f} {row['macro_recall']:>8.4f}")
print('=' * 70)

best = benchmark.iloc[0]
vader_base = benchmark[benchmark['model']=='VADER'].iloc[0]

print(f'\n🏆 Best model        : {best["model"]}')
print(f'   Macro F1          : {best["macro_f1"]:.4f}')
print(f'   Macro Precision   : {best["macro_precision"]:.4f}')
print(f'   Macro Recall      : {best["macro_recall"]:.4f}')
print(f'   Improvement vs VADER: +{best["macro_f1"]-vader_base["macro_f1"]:.4f} F1')
print(f'\nAll models ran successfully on {len(df_test):,} test samples.')
print(f'Random seed: {SEED} (fully reproducible)')

In [ ]:
# ── Styled benchmark DataFrame (publication-quality table) ──────────────
from IPython.display import display

styled_bm = (
    benchmark[['rank','model','type','macro_f1','macro_precision','macro_recall']]
    .copy()
    .rename(columns={'rank':'Rank','model':'Model','type':'Type',
                     'macro_f1':'Macro F1','macro_precision':'Macro Precision',
                     'macro_recall':'Macro Recall'})
)

def row_color(row):
    colors = ['background-color: #0d2b1d; color: #6ee7b7',   # 1st
              'background-color: #0f1e3d; color: #93c5fd',   # 2nd
              'background-color: #1c1033; color: #c4b5fd',   # 3rd
              '',                                             # 4th
              'background-color: #2d0b0b; color: #fca5a5']   # 5th
    idx = int(row['Rank']) - 1
    return [colors[idx]] * len(row)

display(styled_bm.style
    .apply(row_color, axis=1)
    .background_gradient(subset=['Macro F1','Macro Precision','Macro Recall'],
                         cmap='RdYlGn', vmin=0.4, vmax=0.85)
    .format({'Macro F1': '{:.4f}', 'Macro Precision': '{:.4f}', 'Macro Recall': '{:.4f}'})
    .set_caption('📊 Final Benchmark — tweet_eval/sentiment test split')
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size','14px'),('font-weight','bold'),('color','#e2e8f0')]
    }])
    .hide(axis='index')
)

# Plotly table
fig = go.Figure(go.Table(
    header=dict(
        values=['<b>Rank</b>','<b>Model</b>','<b>Type</b>',
                '<b>Macro F1</b>','<b>Macro Precision</b>','<b>Macro Recall</b>'],
        fill_color='#1e293b',
        font=dict(color='white', size=13),
        align='center', height=40
    ),
    cells=dict(
        values=[styled_bm['Rank'], styled_bm['Model'], styled_bm['Type'],
                styled_bm['Macro F1'].round(4),
                styled_bm['Macro Precision'].round(4),
                styled_bm['Macro Recall'].round(4)],
        fill_color=['#0d2b1d','#1e293b','#1e293b','#1e293b','#1e293b','#1e293b'],
        font=dict(color=['#6ee7b7','white','white','#a7f3d0','#93c5fd','#fda4af'], size=12),
        align='center', height=36
    )
))
fig.update_layout(template=PLOTLY_TEMPLATE, height=280,
                  title='Final Benchmark — Interactive Table',
                  margin=dict(l=20,r=20,t=50,b=10))
fig.show()


In [ ]:
# ── Final comparison bar chart ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#0f172a')

bar_w  = 0.25
x      = np.arange(len(benchmark))
model_labels = [m[:10] for m in benchmark['model']]

ax.bar(x - bar_w, benchmark['macro_f1'],        width=bar_w, color='#6366f1', alpha=0.9, label='Macro F1',        edgecolor='none')
ax.bar(x,         benchmark['macro_precision'],  width=bar_w, color='#3b82f6', alpha=0.9, label='Macro Precision', edgecolor='none')
ax.bar(x + bar_w, benchmark['macro_recall'],     width=bar_w, color='#10b981', alpha=0.9, label='Macro Recall',    edgecolor='none')

ax.set_xticks(x); ax.set_xticklabels(model_labels, rotation=20, ha='right', fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Final Model Comparison — Macro F1 · Precision · Recall', color='white', fontsize=14)
ax.legend(facecolor='#1f2937', labelcolor='white', fontsize=11)
ax.axhline(0.7, color='white', linestyle='--', alpha=0.3, linewidth=1)
ax.text(len(benchmark)-0.5, 0.71, '0.7 target', color='gray', fontsize=9)
ax.set_facecolor('#111827')

plt.tight_layout()
plt.show()

print('\n✅ Complete pipeline executed successfully!')
print('   All phases: Data Exploration → Feature Engineering → Classical ML → BiLSTM → Benchmarking')